# 01 - Auditoria inicial de datos

Este notebook documenta la auditoria inicial del dataset `Online Retail II` y muestra el punto de partida metodologico del caso.

Dentro del proyecto, este notebook sirvio para:

- revisar la estructura general del dataset;
- validar tipos de datos y columnas disponibles;
- cuantificar nulos y problemas visibles de calidad;
- dejar documentados los criterios que darian forma a la limpieza de la fase 2.


## Rol de este notebook en el caso

La auditoria se apoya primero en el archivo intermedio generado por la ingesta inicial (`data/interim/online_retail_ii_ingesta_inicial.csv`).

Si ese archivo no existe, el notebook recurre al Excel original en `data/raw/` reutilizando funciones del modulo `src.ingest`.

Con ello, la revision inicial se mantiene alineada con el flujo reproducible del proyecto y evita una lectura aislada del dato.


In [10]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 200)

ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.ingest import cargar_dataset_desde_excel, detectar_archivo_excel

RAW_DIR = ROOT_DIR / "data" / "raw"
INTERIM_PATH = ROOT_DIR / "data" / "interim" / "online_retail_ii_ingesta_inicial.csv"

MAPPING_COLUMNAS = {
    "InvoiceNo": "Invoice",
    "StockCode": "StockCode",
    "Description": "Description",
    "Quantity": "Quantity",
    "InvoiceDate": "InvoiceDate",
    "UnitPrice": "Price",
    "CustomerID": "Customer ID",
    "Country": "Country",
}

COL_INVOICE = MAPPING_COLUMNAS["InvoiceNo"]
COL_STOCK = MAPPING_COLUMNAS["StockCode"]
COL_DESC = MAPPING_COLUMNAS["Description"]
COL_QTY = MAPPING_COLUMNAS["Quantity"]
COL_DATE = MAPPING_COLUMNAS["InvoiceDate"]
COL_PRICE = MAPPING_COLUMNAS["UnitPrice"]
COL_CUSTOMER = MAPPING_COLUMNAS["CustomerID"]
COL_COUNTRY = MAPPING_COLUMNAS["Country"]


In [11]:
if INTERIM_PATH.exists():
    df = pd.read_csv(INTERIM_PATH, parse_dates=[COL_DATE])
    fuente_datos = "interim_csv"
    archivo_origen_auditoria = INTERIM_PATH.name
    hojas_detectadas = sorted(df["origen_hoja"].dropna().unique().tolist()) if "origen_hoja" in df.columns else []
else:
    excel_path = detectar_archivo_excel(None)
    df, hojas_detectadas = cargar_dataset_desde_excel(excel_path)
    fuente_datos = "raw_excel"
    archivo_origen_auditoria = excel_path.name

print(f"Fuente usada en la auditoria: {fuente_datos}")
print(f"Archivo base: {archivo_origen_auditoria}")
print(f"Hojas detectadas: {hojas_detectadas}")
print(f"Shape total: {df.shape}")


Fuente usada en la auditoria: interim_csv
Archivo base: online_retail_ii_ingesta_inicial.csv
Hojas detectadas: ['Year 2009-2010', 'Year 2010-2011']
Shape total: (1067371, 10)


## Aclaracion de nomenclatura

La documentacion oficial del dataset usa nombres como `InvoiceNo`, `UnitPrice` y `CustomerID`.

Sin embargo, el archivo real cargado en este proyecto usa `Invoice`, `Price` y `Customer ID`.

En esta auditoria preservamos los nombres reales del archivo para no alterar la capa `raw` / `interim`, pero dejamos el mapeo explicito para mantener consistencia analitica y documental.


In [12]:
mapeo_columnas = pd.DataFrame(
    [
        {"variable_canonica": canonica, "nombre_en_archivo": real}
        for canonica, real in MAPPING_COLUMNAS.items()
    ]
)
mapeo_columnas


,variable_canonica,nombre_en_archivo
0,InvoiceNo,Invoice
1,StockCode,StockCode
2,Description,Description
3,Quantity,Quantity
4,InvoiceDate,InvoiceDate
5,UnitPrice,Price
6,CustomerID,Customer ID
7,Country,Country


In [13]:
resumen_general = pd.Series(
    {
        "fuente_auditoria": fuente_datos,
        "archivo_base": archivo_origen_auditoria,
        "filas_totales": len(df),
        "columnas_totales": df.shape[1],
        "facturas_unicas": int(df[COL_INVOICE].nunique()) if COL_INVOICE in df.columns else None,
        "clientes_unicos_con_id": int(df[COL_CUSTOMER].dropna().nunique()) if COL_CUSTOMER in df.columns else None,
        "productos_unicos_stockcode": int(df[COL_STOCK].nunique()) if COL_STOCK in df.columns else None,
        "paises_unicos": int(df[COL_COUNTRY].nunique()) if COL_COUNTRY in df.columns else None,
        "fecha_min": df[COL_DATE].min() if COL_DATE in df.columns else None,
        "fecha_max": df[COL_DATE].max() if COL_DATE in df.columns else None,
    },
    name="valor",
).to_frame()
resumen_general


,valor
fuente_auditoria,interim_csv
archivo_base,online_retail_ii_ingesta_inicial.csv
filas_totales,1067371
columnas_totales,10
facturas_unicas,53628
clientes_unicos_con_id,5942
productos_unicos_stockcode,5305
paises_unicos,43
fecha_min,2009-12-01 07:45:00
fecha_max,2011-12-09 12:50:00


In [14]:
if "origen_hoja" in df.columns:
    registros_por_hoja = (
        df["origen_hoja"]
        .value_counts(dropna=False)
        .rename_axis("origen_hoja")
        .reset_index(name="filas")
    )
    registros_por_hoja["porcentaje"] = (registros_por_hoja["filas"] / len(df) * 100).round(2)
    registros_por_hoja
else:
    print("No existe la columna origen_hoja en la fuente cargada.")


In [15]:
columnas_df = pd.DataFrame(
    {
        "columna": df.columns,
        "tipo_dato": df.dtypes.astype(str).values,
    }
)
columnas_df


,columna,tipo_dato
0,Invoice,object
1,StockCode,object
2,Description,object
3,Quantity,int64
4,InvoiceDate,datetime64[ns]
5,Price,float64
6,Customer ID,float64
7,Country,object
8,origen_hoja,object
9,archivo_origen,object


In [16]:
nulos = (
    df.isna()
    .sum()
    .rename("nulos")
    .to_frame()
    .assign(porcentaje_nulos=lambda x: (x["nulos"] / len(df) * 100).round(2))
    .sort_values("nulos", ascending=False)
)
nulos


,nulos,porcentaje_nulos
Customer ID,243007,22.77
Description,4382,0.41
Invoice,0,0.00
StockCode,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00
origen_hoja,0,0.00
archivo_origen,0,0.00


In [17]:
indicadores_calidad = pd.DataFrame(
    [
        {
            "indicador": "CustomerID faltante (Customer ID)",
            "filas": int(df[COL_CUSTOMER].isna().sum()),
        },
        {
            "indicador": "Description faltante",
            "filas": int(df[COL_DESC].isna().sum()),
        },
        {
            "indicador": "InvoiceNo con prefijo C (Invoice)",
            "filas": int(df[COL_INVOICE].astype(str).str.startswith("C", na=False).sum()),
        },
        {
            "indicador": "Quantity negativa",
            "filas": int((df[COL_QTY] < 0).sum()),
        },
        {
            "indicador": "Quantity igual a cero",
            "filas": int((df[COL_QTY] == 0).sum()),
        },
        {
            "indicador": "UnitPrice no positivo (Price)",
            "filas": int((df[COL_PRICE] <= 0).sum()),
        },
        {
            "indicador": "Duplicados exactos",
            "filas": int(df.duplicated().sum()),
        },
    ]
)
indicadores_calidad["porcentaje"] = (indicadores_calidad["filas"] / len(df) * 100).round(2)
indicadores_calidad.sort_values("filas", ascending=False)


,indicador,filas,porcentaje
0,CustomerID faltante (Customer ID),243007,22.77
3,Quantity negativa,22950,2.15
2,InvoiceNo con prefijo C (Invoice),19494,1.83
6,Duplicados exactos,12133,1.14
5,UnitPrice no positivo (Price),6207,0.58
1,Description faltante,4382,0.41
4,Quantity igual a cero,0,0.00


In [18]:
mask_cancelacion = df[COL_INVOICE].astype(str).str.startswith("C", na=False)
mask_quantity_negativa = df[COL_QTY] < 0

cruces_calidad = pd.DataFrame(
    [
        {
            "cruce": "InvoiceNo con prefijo C y Quantity negativa",
            "filas": int((mask_cancelacion & mask_quantity_negativa).sum()),
        },
        {
            "cruce": "InvoiceNo con prefijo C sin Quantity negativa",
            "filas": int((mask_cancelacion & ~mask_quantity_negativa).sum()),
        },
        {
            "cruce": "Quantity negativa sin prefijo C en InvoiceNo",
            "filas": int((~mask_cancelacion & mask_quantity_negativa).sum()),
        },
    ]
)
cruces_calidad


,cruce,filas
0,InvoiceNo con prefijo C y Quantity negativa,19493
1,InvoiceNo con prefijo C sin Quantity negativa,1
2,Quantity negativa sin prefijo C en InvoiceNo,3457


In [19]:
top_paises = (
    df[COL_COUNTRY]
    .value_counts()
    .head(10)
    .rename_axis("Country")
    .reset_index(name="filas")
)
top_paises["porcentaje"] = (top_paises["filas"] / len(df) * 100).round(2)
top_paises


,Country,filas,porcentaje
0,United Kingdom,981330,91.94
1,EIRE,17866,1.67
2,Germany,17624,1.65
3,France,14330,1.34
4,Netherlands,5140,0.48
5,Spain,3811,0.36
6,Switzerland,3189,0.30
7,Belgium,3123,0.29
8,Portugal,2620,0.25
9,Australia,1913,0.18


In [11]:
df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,origen_hoja,archivo_origen
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Year 2009-2010,online_retail_II.xlsx
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,online_retail_II.xlsx
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010,online_retail_II.xlsx
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Year 2009-2010,online_retail_II.xlsx
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Year 2009-2010,online_retail_II.xlsx


## Hallazgos documentados de la corrida actual

Con la corrida ejecutada sobre el dataset disponible en el proyecto, la auditoria inicial deja estos hallazgos que justifican la fase de limpieza posterior:

- Se consolidaron `1,067,371` filas y `10` columnas a partir de dos hojas del Excel original.
- El rango temporal cubre desde `2009-12-01 07:45:00` hasta `2011-12-09 12:50:00`.
- La documentacion oficial del dataset usa `InvoiceNo`, `UnitPrice` y `CustomerID`, mientras que el archivo cargado en esta repo usa `Invoice`, `Price` y `Customer ID`. Esa equivalencia queda explicita en esta auditoria para evitar confusiones posteriores.
- La variable mas critica para analisis de retencion es `CustomerID` (`Customer ID` en el archivo), con `243,007` faltantes (`22.77%`).
- `Description` tiene `4,382` faltantes (`0.41%`), mucho menos severos que los faltantes de cliente.
- Existen `19,494` registros con `InvoiceNo` iniciando en `C` (`Invoice` en el archivo), consistente con la definicion oficial de cancelacion.
- Hay `22,950` filas con `Quantity` negativa (`2.15%`), lo que sugiere devoluciones, ajustes o anulaciones.
- De esas filas, `19,493` coinciden ademas con facturas de prefijo `C`, pero `3,457` tienen cantidad negativa sin ese prefijo, por lo que no conviene asumir una regla unica sin revisar contexto.
- Se detectaron `6,207` filas con `UnitPrice` no positivo (`Price` en el archivo, `0.58%`), que deben revisarse antes de calcular ingresos o ticket.
- Existen `12,133` duplicados exactos (`1.14%`), suficiente volumen como para requerir una validacion explicita en la fase de limpieza.
- `United Kingdom` domina el dataset con `981,330` filas (`91.94%`), por lo que cualquier lectura de negocio agregada estara fuertemente influida por ese mercado.


## Aporte al cierre de la fase 1

A partir de esta auditoria, el proyecto definio los criterios que permitieron cerrar la fase 1 y abrir la fase 2 con una base metodologica clara:

1. Las filas sin `CustomerID` (`Customer ID`) se conservaron en una base general, pero quedaron fuera del analisis customer-level.
2. Las cancelaciones se definieron a partir de `InvoiceNo` (`Invoice`) con prefijo `C`, reforzadas por la validacion de `Quantity`.
3. Los duplicados exactos pasaron a considerarse un ajuste necesario dentro de `processed`.
4. Las filas con `UnitPrice <= 0` quedaron fuera de compras validas y de metricas monetarias.
5. El analisis mantuvo todos los paises y dejo `country` como dimension analitica explicita.
6. Las metricas monetarias se conservaron en GBP, sin conversion de tipo de cambio.
